# 10_graph_readiness_and_artifact_verification.ipynb

## Σκοπός του notebook

Το `NB10` υλοποιείται αυστηρά ως:

**graph data-interface / split-to-graph contract / artifact-verification notebook**

πάνω στο ήδη canonical forecasting-first pipeline του repository.

Ο ρόλος του είναι να επιβεβαιώσει ότι τα υπάρχοντα graph-ready artifacts και τα canonical split artifacts είναι:

- parse-safe
- cohort-safe
- contract-safe
- και graph-ready

για μελλοντική graph-based modeling work.

## Τι κάνει αυτό το notebook

Το `NB10`:

- φορτώνει τα canonical primary artifacts
- εφαρμόζει explicit και stable handling του `park_id`
- επαληθεύει strict timestamp parsing στα canonical splits
- ελέγχει duplicate `(park_id, timestamp)` keys
- επαληθεύει node-order consistency
- επαληθεύει split-to-graph compatibility
- ελέγχει feature / control / blocked-column roles
- κάνει robust optional upstream cohort cross-checks
- και παράγει compact audit / readiness outputs

## Τι δεν κάνει αυτό το notebook

Το `NB10` **δεν είναι**:

- full GNN benchmark notebook
- sequence-model notebook
- νέο benchmark-reporting stage
- νέο feature-engineering stage
- PHM notebook
- νέο diagnostics notebook

Άρα το notebook λειτουργεί ως **strict verification / interface layer** και όχι ως νέο modeling stage.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

# Προαιρετικά graph / connectivity εργαλεία.
# Το notebook δεν εξαρτάται υποχρεωτικά από αυτά για να δουλέψει.
try:
    import torch
    from torch_geometric.utils import coalesce, is_undirected, contains_self_loops
    TORCH_PYG_AVAILABLE = True
except Exception:
    torch = None
    coalesce = None
    is_undirected = None
    contains_self_loops = None
    TORCH_PYG_AVAILABLE = False

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except Exception:
    nx = None
    NETWORKX_AVAILABLE = False

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.6f}".format)

print("NB10 environment initialized.")
print(f"PyG available      : {TORCH_PYG_AVAILABLE}")
print(f"NetworkX available : {NETWORKX_AVAILABLE}")

NB10 environment initialized.
PyG available      : True
NetworkX available : True


In [2]:
# ============================================================
# Project root discovery και optional config import
# ============================================================

def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το project root ανεβαίνοντας από το current working directory.
    Περιμένουμε τουλάχιστον να υπάρχουν οι φάκελοι `data/` και `notebooks/`.
    """
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε project root που να περιέχει `data/` και `notebooks/`."
    )


def resolve_under_root(path_like: str | Path, root: Path) -> Path:
    """
    Αν το path είναι σχετικό, το λύνουμε κάτω από το project root.
    Αν είναι absolute, το κρατάμε ως έχει.
    """
    p = Path(path_like)
    return p if p.is_absolute() else root / p


ROOT = find_project_root(Path.cwd())

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

try:
    import src.config as cfg
    HAS_CFG = True
except Exception:
    cfg = None
    HAS_CFG = False

print(f"Project root          : {ROOT}")
print(f"src.config available  : {HAS_CFG}")

Project root          : C:\Users\diony\Desktop\WindPower_DigitalTwin
src.config available  : True


## Μεθοδολογική θέση του NB10

Το `NB10` έρχεται **μετά** από το ήδη canonical workflow του repository.

Άρα:

- δεν ξανακάνει raw validation
- δεν ξανακάνει split construction
- δεν ξανακάνει benchmark reporting
- δεν ξανακάνει baseline modeling
- δεν συγχέει diagnostics layer με graph-model implementation

Αντίθετα, λειτουργεί ως **strict interface / contract notebook** ανάμεσα στα ήδη canonical artifacts και σε ένα πιθανό future graph-model issue.

## Implemented / immediate next / future work

### Ήδη implemented
- raw validation
- validated-only EDA
- feature engineering
- graph-ready artifact construction
- leakage-aware temporal split
- baseline ladder
- downstream residual diagnostics
- park-level diagnostics / thesis consolidation

### Immediate next step
- strict artifact verification
- split-to-graph compatibility checks
- stable `park_id` / node-order contract
- compact readiness exports

### Future work
- graph-based forecasting models
- sequence models
- GNN / Mamba / Graph-Mamba experimentation

Συνεπώς, το `NB10` παραμένει repository-safe, thesis-ready και χωρίς scope drift.

In [3]:
# ============================================================
# Canonical constants, paths και export policy
# ============================================================

PARK_ID_COLUMN = "park_id"
TIMESTAMP_COLUMN = "timestamp"
FLAG_COLUMN = "test_flag"
TARGET_COLUMN = "Power_Output_Normalized"
BASELINE_COLUMN = "Baseline_Prediction"

PRIMARY_CSV_DTYPES = {
    PARK_ID_COLUMN: "string",
    FLAG_COLUMN: "Int64",
}

PRIMARY_ARTIFACT_PATHS = {
    "train_final": resolve_under_root(
        getattr(cfg, "TRAIN_FINAL_DATASET", "data/processed/train_final.csv")
        if HAS_CFG else "data/processed/train_final.csv",
        ROOT,
    ),
    "val_final": resolve_under_root(
        getattr(cfg, "VAL_FINAL_DATASET", "data/processed/val_final.csv")
        if HAS_CFG else "data/processed/val_final.csv",
        ROOT,
    ),
    "test_final": resolve_under_root(
        getattr(cfg, "TEST_FINAL_DATASET", "data/processed/test_final.csv")
        if HAS_CFG else "data/processed/test_final.csv",
        ROOT,
    ),
    "graph_node_order": resolve_under_root(
        getattr(cfg, "GRAPH_NODE_ORDER_PATH", "data/processed/graph_node_order.csv")
        if HAS_CFG else "data/processed/graph_node_order.csv",
        ROOT,
    ),
    "graph_edge_index": resolve_under_root(
        getattr(cfg, "GRAPH_EDGE_INDEX_PATH", "data/processed/graph_edge_index.npy")
        if HAS_CFG else "data/processed/graph_edge_index.npy",
        ROOT,
    ),
    "graph_distance_matrix_km": resolve_under_root(
        getattr(cfg, "GRAPH_DISTANCE_MATRIX_PATH", "data/processed/graph_distance_matrix_km.npy")
        if HAS_CFG else "data/processed/graph_distance_matrix_km.npy",
        ROOT,
    ),
}

OPTIONAL_ARTIFACT_PATHS = {
    "final_feature_engineered_dataset": resolve_under_root(
        getattr(cfg, "FINAL_DATASET", "data/processed/final_feature_engineered_dataset.csv")
        if HAS_CFG else "data/processed/final_feature_engineered_dataset.csv",
        ROOT,
    ),
    "nb02_meta_coverage_audit": resolve_under_root(
        getattr(cfg, "NB02_META_COVERAGE_AUDIT", "data/processed/nb02_meta_coverage_audit.csv")
        if HAS_CFG else "data/processed/nb02_meta_coverage_audit.csv",
        ROOT,
    ),
    "nb02_nb04_eligibility_summary": resolve_under_root(
        getattr(cfg, "NB02_NB04_ELIGIBILITY_SUMMARY", "data/processed/nb02_nb04_eligibility_summary.csv")
        if HAS_CFG else "data/processed/nb02_nb04_eligibility_summary.csv",
        ROOT,
    ),
    "master_schema_summary": resolve_under_root(
        getattr(cfg, "MASTER_SCHEMA_SUMMARY", "data/processed/master_schema_summary.csv")
        if HAS_CFG else "data/processed/master_schema_summary.csv",
        ROOT,
    ),
}

EXPORT_DIR = ROOT / "data" / "processed" / "diagnostics" / "nb10_graph_contract"

# Προαιρετικό heavy key-space cross-check με το full engineered dataset.
# Από default μένει False ώστε το notebook να μείνει ελαφρύτερο.
RUN_HEAVY_FINAL_DATASET_KEY_CROSSCHECK = False

# Στήλες που δεν πρέπει να θεωρούνται candidate modeling inputs
# χωρίς ρητή μελλοντική συζήτηση.
BLOCKED_MODELING_COLUMNS = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}

print("Primary artifact paths:")
for name, path in PRIMARY_ARTIFACT_PATHS.items():
    print(f"- {name}: {path}")

print("\nOptional artifact paths:")
for name, path in OPTIONAL_ARTIFACT_PATHS.items():
    print(f"- {name}: {path}")

print(f"\nExport dir: {EXPORT_DIR}")

Primary artifact paths:
- train_final: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\train_final.csv
- val_final: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\val_final.csv
- test_final: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\test_final.csv
- graph_node_order: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_node_order.csv
- graph_edge_index: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_edge_index.npy
- graph_distance_matrix_km: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_distance_matrix_km.npy

Optional artifact paths:
- final_feature_engineered_dataset: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\final_feature_engineered_dataset.csv
- nb02_meta_coverage_audit: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_meta_coverage_audit.csv
- nb02_nb04_eligibility_summary: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_nb04_eligibility

In [4]:
# ============================================================
# Core helpers για loading, parsing και robust schema handling
# ============================================================

def ensure_file_exists(path: Path, artifact_name: str) -> None:
    """
    Fail-fast existence check.
    """
    if not path.exists():
        raise FileNotFoundError(f"Λείπει το required artifact `{artifact_name}`: {path}")


def safe_load_csv(
    csv_path: Path,
    artifact_name: str,
    dtype_map: dict[str, str] | None = None,
    usecols: list[str] | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """
    Ασφαλής CSV φόρτωση με deterministic options.
    """
    ensure_file_exists(csv_path, artifact_name)

    return pd.read_csv(
        csv_path,
        dtype=dtype_map,
        usecols=usecols,
        nrows=nrows,
        low_memory=False,
    )


def safe_load_npy(npy_path: Path, artifact_name: str) -> np.ndarray:
    """
    Ασφαλές .npy loading.
    """
    ensure_file_exists(npy_path, artifact_name)

    # Πιο ασφαλής φόρτωση για numeric arrays.
    arr = np.load(npy_path, allow_pickle=False)

    if not isinstance(arr, np.ndarray):
        raise TypeError(f"Το artifact `{artifact_name}` δεν επέστρεψε numpy.ndarray.")

    return arr


def describe_loaded_object(obj: Any) -> str:
    """
    Compact περιγραφή shape / type για manifests.
    """
    if isinstance(obj, pd.DataFrame):
        return f"DataFrame[{obj.shape[0]}x{obj.shape[1]}]"
    if isinstance(obj, np.ndarray):
        return f"ndarray{obj.shape}, dtype={obj.dtype}"
    return str(type(obj))


def build_optional_missing_row(name: str, path: Path) -> dict[str, Any]:
    """
    Manifest row για optional artifact που λείπει.
    """
    return {
        "artifact": name,
        "path": str(path.relative_to(ROOT)),
        "exists": False,
        "parse_ok": False,
        "load_mode": "missing_optional",
        "shape": None,
        "notes": "optional artifact not found",
    }


def standardize_park_id(series: pd.Series) -> pd.Series:
    """
    Μετατρέπει το park_id σε canonical zero-padded string.
    """
    return (
        series.astype("string")
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(5)
    )


def parse_timestamp_strict(series: pd.Series, column_name: str) -> pd.Series:
    """
    Strict datetime parsing για canonical downstream backbone columns.
    """
    parsed = pd.to_datetime(
        series,
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )

    if parsed.isna().any():
        bad_count = int(parsed.isna().sum())
        raise ValueError(
            f"Αποτυχία strict datetime parsing στη στήλη '{column_name}'. "
            f"NaT count = {bad_count}"
        )

    return parsed


def ensure_required_columns(
    df: pd.DataFrame,
    required_columns: set[str],
    df_name: str,
) -> None:
    """
    Fail-fast έλεγχος required columns.
    """
    missing = sorted(required_columns - set(df.columns))
    if missing:
        raise KeyError(f"Λείπουν required columns από το `{df_name}`: {missing}")


def assert_no_duplicate_keys(
    df: pd.DataFrame,
    key_cols: list[str],
    df_name: str,
) -> None:
    """
    Δεν επιτρέπουμε duplicate rows πάνω στα backbone keys.
    """
    dup_count = int(df.duplicated(subset=key_cols).sum())
    if dup_count > 0:
        raise ValueError(
            f"Βρέθηκαν {dup_count} duplicate rows στο `{df_name}` "
            f"πάνω στα keys {key_cols}."
        )


def assert_no_key_overlap(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_name: str,
    right_name: str,
    key_cols: list[str],
) -> None:
    """
    Ελέγχει ότι δύο splits δεν έχουν overlap στο ίδιο backbone key space.
    """
    overlap = left_df[key_cols].merge(right_df[key_cols], on=key_cols, how="inner")

    if not overlap.empty:
        raise ValueError(
            f"Βρέθηκε overlap μεταξύ `{left_name}` και `{right_name}` "
            f"πάνω στα keys {key_cols}. Overlap rows = {len(overlap)}"
        )


def normalize_column_name(col: str) -> str:
    """
    Κανονικοποιεί column names για πιο robust alias matching.
    """
    return (
        str(col)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def find_first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Επιστρέφει το πρώτο column που υπάρχει είτε ακριβώς είτε μέσω normalized matching.
    """
    # Πρώτα exact match
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    # Μετά normalized match
    normalized_to_original = {
        normalize_column_name(col): col
        for col in df.columns
    }

    for candidate in candidates:
        normalized_candidate = normalize_column_name(candidate)
        if normalized_candidate in normalized_to_original:
            return normalized_to_original[normalized_candidate]

    return None


def to_bool_like(series: pd.Series) -> pd.Series:
    """
    Μετατρέπει διάφορες boolean-like αναπαραστάσεις σε καθαρό bool Series.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    true_values = {"true", "1", "yes", "y"}
    return normalized.isin(true_values)

## Φόρτωση primary και optional artifacts

Σε αυτό το βήμα:

- τα **primary artifacts** είναι υποχρεωτικά και fail-fast
- τα **optional artifacts** φορτώνονται μόνο για cross-check / schema audit
- το `final_feature_engineered_dataset.csv` φορτώνεται αρχικά **header-only**
  για να παραμείνει το notebook πιο ελαφρύ
- το notebook **δεν** ανοίγει benchmark / prediction artifacts γιατί ο ρόλος του είναι strict contract verification
- το notebook δουλεύει πάνω στα ήδη υπάρχοντα canonical local rerun artifacts και τα **επαληθεύει**
  αντί να τα ξαναδημιουργεί

Σημείωση policy:

- δεν χρειάζεται διαγραφή των canonical processed CSVs πριν από το rerun του `NB10`
- μόνο τα compact NB10 exports μπορούν, αν χρειαστεί, να γίνουν overwrite στο τέλος του notebook

In [5]:
# ============================================================
# Load primary artifacts
# ============================================================

artifact_status_rows: list[dict[str, Any]] = []

train_raw = safe_load_csv(
    PRIMARY_ARTIFACT_PATHS["train_final"],
    "train_final",
    dtype_map=PRIMARY_CSV_DTYPES,
)
artifact_status_rows.append(
    {
        "artifact": "train_final",
        "path": str(PRIMARY_ARTIFACT_PATHS["train_final"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "full_csv",
        "shape": describe_loaded_object(train_raw),
        "notes": None,
    }
)

val_raw = safe_load_csv(
    PRIMARY_ARTIFACT_PATHS["val_final"],
    "val_final",
    dtype_map=PRIMARY_CSV_DTYPES,
)
artifact_status_rows.append(
    {
        "artifact": "val_final",
        "path": str(PRIMARY_ARTIFACT_PATHS["val_final"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "full_csv",
        "shape": describe_loaded_object(val_raw),
        "notes": None,
    }
)

test_raw = safe_load_csv(
    PRIMARY_ARTIFACT_PATHS["test_final"],
    "test_final",
    dtype_map=PRIMARY_CSV_DTYPES,
)
artifact_status_rows.append(
    {
        "artifact": "test_final",
        "path": str(PRIMARY_ARTIFACT_PATHS["test_final"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "full_csv",
        "shape": describe_loaded_object(test_raw),
        "notes": None,
    }
)

graph_node_order_raw = safe_load_csv(
    PRIMARY_ARTIFACT_PATHS["graph_node_order"],
    "graph_node_order",
)
artifact_status_rows.append(
    {
        "artifact": "graph_node_order",
        "path": str(PRIMARY_ARTIFACT_PATHS["graph_node_order"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "full_csv",
        "shape": describe_loaded_object(graph_node_order_raw),
        "notes": None,
    }
)

graph_edge_index = safe_load_npy(
    PRIMARY_ARTIFACT_PATHS["graph_edge_index"],
    "graph_edge_index",
)
artifact_status_rows.append(
    {
        "artifact": "graph_edge_index",
        "path": str(PRIMARY_ARTIFACT_PATHS["graph_edge_index"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "npy",
        "shape": describe_loaded_object(graph_edge_index),
        "notes": None,
    }
)

graph_distance_matrix_km = safe_load_npy(
    PRIMARY_ARTIFACT_PATHS["graph_distance_matrix_km"],
    "graph_distance_matrix_km",
)
artifact_status_rows.append(
    {
        "artifact": "graph_distance_matrix_km",
        "path": str(PRIMARY_ARTIFACT_PATHS["graph_distance_matrix_km"].relative_to(ROOT)),
        "exists": True,
        "parse_ok": True,
        "load_mode": "npy",
        "shape": describe_loaded_object(graph_distance_matrix_km),
        "notes": None,
    }
)

# ============================================================
# Load optional artifacts
# ============================================================

# 1) final_feature_engineered_dataset.csv -> header-only load
if OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"].exists():
    engineered_schema_df = safe_load_csv(
        OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"],
        "final_feature_engineered_dataset",
        nrows=0,
    )
    artifact_status_rows.append(
        {
            "artifact": "final_feature_engineered_dataset",
            "path": str(OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"].relative_to(ROOT)),
            "exists": True,
            "parse_ok": True,
            "load_mode": "header_only_csv",
            "shape": describe_loaded_object(engineered_schema_df),
            "notes": "header-only schema load",
        }
    )
else:
    engineered_schema_df = None
    artifact_status_rows.append(
        build_optional_missing_row(
            "final_feature_engineered_dataset",
            OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"],
        )
    )

# 2) nb02_meta_coverage_audit.csv -> detailed park-level upstream audit
if OPTIONAL_ARTIFACT_PATHS["nb02_meta_coverage_audit"].exists():
    meta_coverage_audit_df = safe_load_csv(
        OPTIONAL_ARTIFACT_PATHS["nb02_meta_coverage_audit"],
        "nb02_meta_coverage_audit",
    )
    artifact_status_rows.append(
        {
            "artifact": "nb02_meta_coverage_audit",
            "path": str(OPTIONAL_ARTIFACT_PATHS["nb02_meta_coverage_audit"].relative_to(ROOT)),
            "exists": True,
            "parse_ok": True,
            "load_mode": "full_csv",
            "shape": describe_loaded_object(meta_coverage_audit_df),
            "notes": "detailed upstream park-level audit",
        }
    )
else:
    meta_coverage_audit_df = None
    artifact_status_rows.append(
        build_optional_missing_row(
            "nb02_meta_coverage_audit",
            OPTIONAL_ARTIFACT_PATHS["nb02_meta_coverage_audit"],
        )
    )

# 3) nb02_nb04_eligibility_summary.csv -> summary-only reference
if OPTIONAL_ARTIFACT_PATHS["nb02_nb04_eligibility_summary"].exists():
    eligibility_summary_df = safe_load_csv(
        OPTIONAL_ARTIFACT_PATHS["nb02_nb04_eligibility_summary"],
        "nb02_nb04_eligibility_summary",
    )
    artifact_status_rows.append(
        {
            "artifact": "nb02_nb04_eligibility_summary",
            "path": str(OPTIONAL_ARTIFACT_PATHS["nb02_nb04_eligibility_summary"].relative_to(ROOT)),
            "exists": True,
            "parse_ok": True,
            "load_mode": "full_csv",
            "shape": describe_loaded_object(eligibility_summary_df),
            "notes": "summary artifact only",
        }
    )
else:
    eligibility_summary_df = None
    artifact_status_rows.append(
        build_optional_missing_row(
            "nb02_nb04_eligibility_summary",
            OPTIONAL_ARTIFACT_PATHS["nb02_nb04_eligibility_summary"],
        )
    )

# 4) master_schema_summary.csv -> optional schema reference
if OPTIONAL_ARTIFACT_PATHS["master_schema_summary"].exists():
    master_schema_summary_df = safe_load_csv(
        OPTIONAL_ARTIFACT_PATHS["master_schema_summary"],
        "master_schema_summary",
    )
    artifact_status_rows.append(
        {
            "artifact": "master_schema_summary",
            "path": str(OPTIONAL_ARTIFACT_PATHS["master_schema_summary"].relative_to(ROOT)),
            "exists": True,
            "parse_ok": True,
            "load_mode": "full_csv",
            "shape": describe_loaded_object(master_schema_summary_df),
            "notes": None,
        }
    )
else:
    master_schema_summary_df = None
    artifact_status_rows.append(
        build_optional_missing_row(
            "master_schema_summary",
            OPTIONAL_ARTIFACT_PATHS["master_schema_summary"],
        )
    )

artifact_status_manifest_df = pd.DataFrame(artifact_status_rows)

print("Primary + optional artifact loading completed.")
display(artifact_status_manifest_df)

Primary + optional artifact loading completed.


,artifact,path,exists,parse_ok,load_mode,shape,notes
0,train_final,data\processed\train_final.csv,True,True,full_csv,DataFrame[1982736x47],NaN
1,val_final,data\processed\val_final.csv,True,True,full_csv,DataFrame[182998x47],NaN
2,test_final,data\processed\test_final.csv,True,True,full_csv,DataFrame[1086336x47],NaN
3,graph_node_order,data\processed\graph_node_order.csv,True,True,full_csv,DataFrame[256x1],NaN
4,graph_edge_index,data\processed\graph_edge_index.npy,True,True,npy,"ndarray(2, 1068), dtype=int64",NaN
5,graph_distance_matrix_km,data\processed\graph_distance_matrix_km.npy,True,True,npy,"ndarray(256, 256), dtype=float64",NaN
6,final_feature_engineered_dataset,data\processed\final_feature_engineered_datase...,True,True,header_only_csv,DataFrame[0x47],header-only schema load
7,nb02_meta_coverage_audit,data\processed\nb02_meta_coverage_audit.csv,True,True,full_csv,DataFrame[272x47],detailed upstream park-level audit
8,nb02_nb04_eligibility_summary,data\processed\nb02_nb04_eligibility_summary.csv,True,True,full_csv,DataFrame[2x2],summary artifact only
9,master_schema_summary,data\processed\master_schema_summary.csv,True,True,full_csv,DataFrame[40x5],NaN


## Node-order contract και split-side backbone audit

Το πιο κρίσιμο verification βήμα του `NB10` είναι να αποδείξει ότι:

1. το `graph_node_order.csv` ορίζει καθαρά το node identity space
2. τα canonical splits έχουν υγιές backbone key space
3. δεν υπάρχουν duplicate `(park_id, timestamp)` rows
4. δεν υπάρχει overlap μεταξύ `train / val / test`
5. και όλα τα split parks είναι συμβατά με το graph node order

In [6]:
# ============================================================
# A. Node-order audit
# ============================================================

# Επιτρέπουμε λίγο πιο robust alias handling μόνο εδώ,
# ώστε το notebook να μη σπάει αν το node-order export χρησιμοποιεί
# μικρή παραλλαγή στο όνομα της στήλης.
graph_node_id_col = find_first_existing_column(
    graph_node_order_raw,
    ["park_id", "loc_id", "location_id", "park"],
)

if graph_node_id_col is None:
    raise KeyError(
        "Δεν βρέθηκε park-id-like στήλη στο graph_node_order.csv. "
        f"Available columns: {graph_node_order_raw.columns.tolist()}"
    )

graph_node_order_df = graph_node_order_raw.copy()
graph_node_order_df[PARK_ID_COLUMN] = standardize_park_id(graph_node_order_df[graph_node_id_col])

if graph_node_order_df[PARK_ID_COLUMN].isna().any():
    raise ValueError("Βρέθηκαν null park_id values στο graph_node_order.csv.")

if graph_node_order_df.columns.duplicated().any():
    duplicated_cols = graph_node_order_df.columns[graph_node_order_df.columns.duplicated()].tolist()
    raise ValueError(f"Βρέθηκαν duplicate column names στο graph_node_order.csv: {duplicated_cols}")

dup_nodes = int(graph_node_order_df.duplicated(subset=[PARK_ID_COLUMN]).sum())
if dup_nodes > 0:
    raise ValueError(f"Βρέθηκαν {dup_nodes} duplicate park_id rows στο graph_node_order.csv.")

graph_node_order_df = graph_node_order_df.reset_index(drop=True).copy()
graph_node_order_df["node_idx"] = np.arange(len(graph_node_order_df), dtype=int)

graph_node_set = set(graph_node_order_df[PARK_ID_COLUMN].astype(str))
node_index_map_df = graph_node_order_df[["node_idx", PARK_ID_COLUMN]].copy()
node_index_lookup = dict(zip(node_index_map_df[PARK_ID_COLUMN], node_index_map_df["node_idx"]))

print("Node-order audit completed.")
print(f"n_nodes = {len(graph_node_order_df)}")
display(node_index_map_df.head())

# ============================================================
# B. Split-side audit helper
# ============================================================

REQUIRED_SPLIT_COLUMNS = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}

def audit_split_dataframe(
    df_raw: pd.DataFrame,
    split_name: str,
    expected_flag_values: set[int],
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Καθαρίζει / ελέγχει το canonical split dataframe και επιστρέφει:
    - normalized dataframe
    - compact audit summary row
    """
    df = df_raw.copy()

    ensure_required_columns(df, REQUIRED_SPLIT_COLUMNS, split_name)

    if df.columns.duplicated().any():
        duplicated_cols = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(
            f"Βρέθηκαν duplicate column names στο split `{split_name}`: {duplicated_cols}"
        )

    # Ρητή canonical normalization.
    df[PARK_ID_COLUMN] = standardize_park_id(df[PARK_ID_COLUMN])
    df[TIMESTAMP_COLUMN] = parse_timestamp_strict(df[TIMESTAMP_COLUMN], TIMESTAMP_COLUMN)

    # Core null checks.
    core_nulls = df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, FLAG_COLUMN, TARGET_COLUMN]].isnull().sum()
    if int(core_nulls.sum()) > 0:
        raise ValueError(
            f"Βρέθηκαν nulls στα core columns του split `{split_name}`:\n"
            f"{core_nulls.to_string()}"
        )

    # Duplicate backbone keys.
    assert_no_duplicate_keys(
        df,
        [PARK_ID_COLUMN, TIMESTAMP_COLUMN],
        split_name,
    )

    # Deterministic ordering.
    df = (
        df.sort_values([PARK_ID_COLUMN, TIMESTAMP_COLUMN], kind="mergesort")
          .reset_index(drop=True)
          .copy()
    )

    # Monotonic timestamp check ανά park.
    monotonic_ok = (
        df.groupby(PARK_ID_COLUMN, sort=False)[TIMESTAMP_COLUMN]
          .apply(lambda s: bool(s.is_monotonic_increasing))
          .all()
    )
    if not monotonic_ok:
        raise ValueError(
            f"Το split `{split_name}` δεν έχει monotonic timestamp ordering ανά park_id."
        )

    # test_flag contract.
    found_flags = set(
        pd.Series(df[FLAG_COLUMN].dropna().unique()).astype(int).tolist()
    )
    if found_flags != expected_flag_values:
        raise ValueError(
            f"Παραβίαση test_flag contract στο `{split_name}`. "
            f"Expected={sorted(expected_flag_values)}, found={sorted(found_flags)}"
        )

    if len(df) == 0:
        raise ValueError(f"Το split `{split_name}` είναι κενό.")

    split_park_set = set(df[PARK_ID_COLUMN].astype(str))
    missing_from_graph = sorted(split_park_set - graph_node_set)
    extra_vs_graph = sorted(graph_node_set - split_park_set)

    summary = {
        "split": split_name,
        "rows": len(df),
        "cols": df.shape[1],
        "unique_parks": df[PARK_ID_COLUMN].nunique(),
        "unique_timestamps": df[TIMESTAMP_COLUMN].nunique(),
        "timestamp_min": df[TIMESTAMP_COLUMN].min(),
        "timestamp_max": df[TIMESTAMP_COLUMN].max(),
        "duplicate_keys": 0,
        "flag_values": sorted(found_flags),
        "parks_missing_from_graph": len(missing_from_graph),
        "parks_extra_vs_graph": len(extra_vs_graph),
        "all_parks_in_graph": len(missing_from_graph) == 0,
    }

    return df, summary


train_df, train_summary = audit_split_dataframe(train_raw, "train", {0})
val_df, val_summary = audit_split_dataframe(val_raw, "val", {0})
test_df, test_summary = audit_split_dataframe(test_raw, "test", {1})

# ============================================================
# C. Cross-split schema και overlap checks
# ============================================================

train_columns = list(train_df.columns)
val_columns = list(val_df.columns)
test_columns = list(test_df.columns)

if train_columns != val_columns or train_columns != test_columns:
    raise ValueError("Το schema δεν είναι identical across train / val / test.")

assert_no_key_overlap(
    train_df,
    val_df,
    "train",
    "val",
    [PARK_ID_COLUMN, TIMESTAMP_COLUMN],
)
assert_no_key_overlap(
    train_df,
    test_df,
    "train",
    "test",
    [PARK_ID_COLUMN, TIMESTAMP_COLUMN],
)
assert_no_key_overlap(
    val_df,
    test_df,
    "val",
    "test",
    [PARK_ID_COLUMN, TIMESTAMP_COLUMN],
)

split_audit_df = pd.DataFrame([train_summary, val_summary, test_summary])

print("Split-side backbone audit completed.")
display(split_audit_df)

Node-order audit completed.
n_nodes = 256


,node_idx,park_id
0,0,00011
1,1,00090
2,2,00161
3,3,00164
4,4,00183


Split-side backbone audit completed.


,split,rows,cols,unique_parks,unique_timestamps,timestamp_min,timestamp_max,duplicate_keys,flag_values,parks_missing_from_graph,parks_extra_vs_graph,all_parks_in_graph
0,train,1982736,47,256,7819,2018-12-08 06:00:00,2019-11-01 00:00:00,0,[0],0,0,True
1,val,182998,47,256,720,2019-11-01 01:00:00,2019-12-01 00:00:00,0,[0],0,0,True
2,test,1086336,47,256,4295,2019-12-01 01:00:00,2020-06-01 23:00:00,0,[1],0,0,True


## Split-to-graph contract και graph structure audit

Εδώ ελέγχεται αν:

- κάθε split row μπορεί να αντιστοιχιστεί σε valid `node_idx`
- το union των split parks συμφωνεί με το graph node cohort
- το `graph_edge_index.npy` είναι shape-valid, dtype-valid και index-valid
- υπάρχουν self-loops, duplicate directed edges ή reciprocal inconsistencies

Σημαντική methodological σημείωση:

Το notebook **δεν διορθώνει** τα observed temporal windows των local artifacts.
Αν το current local contract δείξει test-start γύρω στην `2019-12-01`, αυτό πρέπει να καταγραφεί ως **verification finding**
και όχι να κρυφτεί ή να αλλάξει ad hoc μέσα στο notebook.

In [7]:
# ============================================================
# A. Split-to-graph mapping checks
# ============================================================

for df_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["node_idx"] = df[PARK_ID_COLUMN].map(node_index_lookup)

    unmapped_rows = int(df["node_idx"].isna().sum())
    if unmapped_rows > 0:
        raise ValueError(
            f"Βρέθηκαν {unmapped_rows} unmapped rows στο split `{df_name}`."
        )

    df["node_idx"] = df["node_idx"].astype(int)

split_graph_contract_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "unique_parks": train_df[PARK_ID_COLUMN].nunique(),
            "all_rows_mappable_to_node_index": bool(train_df["node_idx"].notna().all()),
            "parks_missing_from_graph": int(len(set(train_df[PARK_ID_COLUMN]) - graph_node_set)),
            "timestamp_min": train_df[TIMESTAMP_COLUMN].min(),
            "timestamp_max": train_df[TIMESTAMP_COLUMN].max(),
        },
        {
            "split": "val",
            "rows": len(val_df),
            "unique_parks": val_df[PARK_ID_COLUMN].nunique(),
            "all_rows_mappable_to_node_index": bool(val_df["node_idx"].notna().all()),
            "parks_missing_from_graph": int(len(set(val_df[PARK_ID_COLUMN]) - graph_node_set)),
            "timestamp_min": val_df[TIMESTAMP_COLUMN].min(),
            "timestamp_max": val_df[TIMESTAMP_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "unique_parks": test_df[PARK_ID_COLUMN].nunique(),
            "all_rows_mappable_to_node_index": bool(test_df["node_idx"].notna().all()),
            "parks_missing_from_graph": int(len(set(test_df[PARK_ID_COLUMN]) - graph_node_set)),
            "timestamp_min": test_df[TIMESTAMP_COLUMN].min(),
            "timestamp_max": test_df[TIMESTAMP_COLUMN].max(),
        },
    ]
)

split_union_park_set = (
    set(train_df[PARK_ID_COLUMN].astype(str))
    | set(val_df[PARK_ID_COLUMN].astype(str))
    | set(test_df[PARK_ID_COLUMN].astype(str))
)

cohort_contract_summary_rows = [
    {
        "check": "split_union_equals_graph_nodes",
        "status": split_union_park_set == graph_node_set,
        "details": (
            "PASS"
            if split_union_park_set == graph_node_set
            else f"graph_only={len(graph_node_set - split_union_park_set)}, "
                 f"split_only={len(split_union_park_set - graph_node_set)}"
        ),
    }
]

display(split_graph_contract_summary_df)

# ============================================================
# B. edge_index audit
# ============================================================

edge_index_arr = np.asarray(graph_edge_index)

if edge_index_arr.ndim != 2:
    raise ValueError(
        f"Το graph_edge_index.npy πρέπει να είναι 2D array. Found shape={edge_index_arr.shape}"
    )

if edge_index_arr.shape[0] != 2:
    raise ValueError(
        "Το graph_edge_index.npy δεν είναι σε canonical COO μορφή [2, num_edges]. "
        f"Found shape={edge_index_arr.shape}"
    )

if edge_index_arr.shape[1] == 0:
    raise ValueError("Το graph_edge_index.npy είναι κενό.")

if not np.issubdtype(edge_index_arr.dtype, np.integer):
    raise TypeError(
        f"Το graph_edge_index.npy πρέπει να έχει integer dtype. Found={edge_index_arr.dtype}"
    )

if not np.isfinite(edge_index_arr).all():
    raise ValueError("Το graph_edge_index.npy περιέχει non-finite values.")

n_nodes = len(graph_node_order_df)
src = edge_index_arr[0]
dst = edge_index_arr[1]

min_idx = int(edge_index_arr.min())
max_idx = int(edge_index_arr.max())

if min_idx < 0:
    raise ValueError(f"Βρέθηκαν αρνητικά node indices στο edge_index. min={min_idx}")

if max_idx >= n_nodes:
    raise ValueError(
        f"Βρέθηκαν out-of-range node indices στο edge_index. "
        f"max={max_idx}, n_nodes={n_nodes}"
    )

edge_df = pd.DataFrame({"src": src, "dst": dst})
duplicate_directed_edges = int(edge_df.duplicated().sum())
self_loops_count = int((src == dst).sum())

pair_set = set(map(tuple, edge_df[["src", "dst"]].to_records(index=False)))
reciprocal_missing_count = 0
for u, v in pair_set:
    if u != v and (v, u) not in pair_set:
        reciprocal_missing_count += 1

undirected_edge_df = pd.DataFrame(
    {
        "u": np.minimum(src, dst),
        "v": np.maximum(src, dst),
    }
)
unique_undirected_edges = int(len(undirected_edge_df.drop_duplicates()))

graph_artifact_summary = {
    "n_nodes": n_nodes,
    "n_directed_edges": int(edge_index_arr.shape[1]),
    "n_unique_undirected_edges": unique_undirected_edges,
    "self_loops": self_loops_count,
    "duplicate_directed_edges": duplicate_directed_edges,
    "reciprocal_missing_count": reciprocal_missing_count,
    "edge_index_min": min_idx,
    "edge_index_max": max_idx,
}

# Προαιρετικός PyG audit.
if TORCH_PYG_AVAILABLE:
    edge_index_t = torch.as_tensor(edge_index_arr, dtype=torch.long)
    coalesced_edge_index = coalesce(edge_index_t, num_nodes=n_nodes)

    graph_artifact_summary["pyg_is_undirected"] = bool(
        is_undirected(edge_index_t, num_nodes=n_nodes)
    )
    graph_artifact_summary["pyg_contains_self_loops"] = bool(
        contains_self_loops(edge_index_t)
    )
    graph_artifact_summary["pyg_coalesced_edge_count"] = int(
        coalesced_edge_index.size(1)
    )
else:
    graph_artifact_summary["pyg_is_undirected"] = None
    graph_artifact_summary["pyg_contains_self_loops"] = None
    graph_artifact_summary["pyg_coalesced_edge_count"] = None

# Προαιρετικός connectivity audit.
if NETWORKX_AVAILABLE:
    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    G.add_edges_from(
        undirected_edge_df.drop_duplicates()[["u", "v"]].itertuples(index=False, name=None)
    )

    component_sizes = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    graph_artifact_summary["n_connected_components"] = int(nx.number_connected_components(G))
    graph_artifact_summary["largest_component_size"] = int(component_sizes[0]) if component_sizes else 0
    graph_artifact_summary["largest_component_share"] = (
        (component_sizes[0] / n_nodes) if component_sizes and n_nodes > 0 else None
    )
else:
    graph_artifact_summary["n_connected_components"] = None
    graph_artifact_summary["largest_component_size"] = None
    graph_artifact_summary["largest_component_share"] = None

graph_artifact_summary_df = pd.DataFrame([graph_artifact_summary])

print("Graph structure audit completed.")
display(graph_artifact_summary_df)

,split,rows,unique_parks,all_rows_mappable_to_node_index,parks_missing_from_graph,timestamp_min,timestamp_max
0,train,1982736,256,True,0,2018-12-08 06:00:00,2019-11-01 00:00:00
1,val,182998,256,True,0,2019-11-01 01:00:00,2019-12-01 00:00:00
2,test,1086336,256,True,0,2019-12-01 01:00:00,2020-06-01 23:00:00


Graph structure audit completed.


,n_nodes,n_directed_edges,n_unique_undirected_edges,self_loops,duplicate_directed_edges,reciprocal_missing_count,edge_index_min,edge_index_max,pyg_is_undirected,pyg_contains_self_loops,pyg_coalesced_edge_count,n_connected_components,largest_component_size,largest_component_share
0,256,1068,534,0,0,0,0,255,True,False,1068,2,251,0.980469


## Distance-matrix audit, feature-role manifest και upstream optional cross-checks

Σε αυτό το βήμα:

- ελέγχουμε το `graph_distance_matrix_km.npy`
- χτίζουμε ένα compact feature-role manifest
- και κάνουμε **robust optional upstream cross-checks**

Σημαντικό:

- το detailed park-level eligibility check προσπαθεί πρώτα να χρησιμοποιήσει το
  `nb02_meta_coverage_audit.csv`
- το `nb02_nb04_eligibility_summary.csv` αντιμετωπίζεται ως summary artifact
  και **όχι** ως υποχρεωτικό park-level contract source
- έτσι αποφεύγουμε το προηγούμενο σφάλμα όπου γινόταν λανθασμένη υπόθεση ότι
  το summary artifact έχει οπωσδήποτε στήλη `park_id`

Επιπλέον, αν το observed test-start των current local artifacts πέσει ξανά πάνω στην `2019-12-01`,
το εύρημα αυτό είναι συμβατό με τη documented dataset interpretation του Vogt και πρέπει να μείνει ορατό ως verification result.

In [8]:
# ============================================================
# A. Distance-matrix audit
# ============================================================

distance_arr = np.asarray(graph_distance_matrix_km)

if distance_arr.ndim != 2:
    raise ValueError(
        f"Το graph_distance_matrix_km.npy πρέπει να είναι 2D. Found shape={distance_arr.shape}"
    )

if distance_arr.shape[0] != distance_arr.shape[1]:
    raise ValueError(
        "Το graph_distance_matrix_km.npy πρέπει να είναι square matrix. "
        f"Found shape={distance_arr.shape}"
    )

if distance_arr.shape[0] != n_nodes:
    raise ValueError(
        "Το graph_distance_matrix_km.npy δεν συμφωνεί με το node count. "
        f"distance_nodes={distance_arr.shape[0]}, n_nodes={n_nodes}"
    )

if not np.isfinite(distance_arr).all():
    raise ValueError("Το graph_distance_matrix_km.npy περιέχει non-finite values.")

if (distance_arr < 0).any():
    raise ValueError("Το graph_distance_matrix_km.npy περιέχει αρνητικές αποστάσεις.")

diag = np.diag(distance_arr)
diagonal_near_zero = bool(np.allclose(diag, 0.0, atol=1e-9))
symmetric_ok = bool(np.allclose(distance_arr, distance_arr.T, atol=1e-9))

distance_summary_df = pd.DataFrame(
    [
        {
            "distance_shape": distance_arr.shape,
            "distance_min": float(distance_arr.min()),
            "distance_max": float(distance_arr.max()),
            "diagonal_near_zero": diagonal_near_zero,
            "symmetric_ok": symmetric_ok,
        }
    ]
)

print("Distance-matrix audit completed.")
display(distance_summary_df)

# ============================================================
# B. Feature-role manifest
# ============================================================

if engineered_schema_df is not None:
    schema_columns = list(engineered_schema_df.columns)
else:
    schema_columns = list(train_df.columns)

def classify_feature_role(col: str) -> str:
    """
    Ρητή και συντηρητική ταξινόμηση στηλών για future graph-model handoff.
    """
    if col == TARGET_COLUMN:
        return "target"
    if col in {PARK_ID_COLUMN, TIMESTAMP_COLUMN, FLAG_COLUMN}:
        return "backbone_control"
    if col == BASELINE_COLUMN:
        return "control_baseline"
    if col.startswith("nb02_"):
        return "provenance_audit"
    if "audit" in col.lower() or "status" in col.lower() or "coverage" in col.lower():
        return "provenance_audit"
    if col in BLOCKED_MODELING_COLUMNS:
        return "blocked_modeling_column"
    return "candidate_feature"

feature_role_manifest_df = pd.DataFrame(
    {
        "column": schema_columns,
        "role": [classify_feature_role(col) for col in schema_columns],
        "present_in_train": [col in train_df.columns for col in schema_columns],
        "numeric_in_train": [
            (col in train_df.columns and pd.api.types.is_numeric_dtype(train_df[col]))
            for col in schema_columns
        ],
    }
)

print("Feature-role manifest created.")
display(feature_role_manifest_df.head(25))

# ============================================================
# C. Optional eligibility cross-check
# ============================================================

audit_source_notes = []

if meta_coverage_audit_df is not None:
    # Προσπαθούμε να βρούμε park-level id column.
    meta_park_col = find_first_existing_column(
        meta_coverage_audit_df,
        ["park_id", "loc_id", "location_id", "park"],
    )

    if meta_park_col is not None:
        eligibility_work = meta_coverage_audit_df.copy()
        eligibility_work[PARK_ID_COLUMN] = standardize_park_id(eligibility_work[meta_park_col])

        possible_eligibility_cols = [
            "nb04_eligible",
            "is_nb04_eligible",
            "eligible",
            "strict_downstream_eligible",
            "downstream_eligible",
        ]
        elig_col = find_first_existing_column(eligibility_work, possible_eligibility_cols)

        if elig_col is not None:
            eligibility_work["nb10_eligibility_bool"] = to_bool_like(
                eligibility_work[elig_col]
            )

            eligible_park_set = set(
                eligibility_work.loc[
                    eligibility_work["nb10_eligibility_bool"],
                    PARK_ID_COLUMN
                ].astype(str)
            )

            cohort_contract_summary_rows.append(
                {
                    "check": "eligible_parks_equal_graph_nodes",
                    "status": eligible_park_set == graph_node_set,
                    "details": (
                        "PASS"
                        if eligible_park_set == graph_node_set
                        else f"graph_only={len(graph_node_set - eligible_park_set)}, "
                             f"eligible_only={len(eligible_park_set - graph_node_set)}"
                    ),
                }
            )

            audit_source_notes.append(
                f"Eligibility cross-check used `nb02_meta_coverage_audit.csv` "
                f"with park column `{meta_park_col}` and eligibility column `{elig_col}`."
            )
        else:
            cohort_contract_summary_rows.append(
                {
                    "check": "eligible_parks_equal_graph_nodes",
                    "status": None,
                    "details": "park-level audit found, but no eligibility-like column was found",
                }
            )
            audit_source_notes.append(
                "Detailed upstream audit exists, αλλά δεν βρέθηκε eligibility-like column."
            )
    else:
        cohort_contract_summary_rows.append(
            {
                "check": "eligible_parks_equal_graph_nodes",
                "status": None,
                "details": "detailed audit exists, but no park-id-like column was found",
            }
        )
        audit_source_notes.append(
            "Detailed upstream audit exists, αλλά δεν βρέθηκε park-id-like column."
        )

elif eligibility_summary_df is not None:
    # Το summary artifact παραμένει χρήσιμο ως reference,
    # αλλά δεν το χρησιμοποιούμε για park-level set equality χωρίς
    # ρητή ύπαρξη park-id-like column.
    summary_park_col = find_first_existing_column(
        eligibility_summary_df,
        ["park_id", "loc_id", "location_id", "park"],
    )

    if summary_park_col is not None:
        summary_work = eligibility_summary_df.copy()
        summary_work[PARK_ID_COLUMN] = standardize_park_id(summary_work[summary_park_col])

        possible_eligibility_cols = [
            "nb04_eligible",
            "is_nb04_eligible",
            "eligible",
            "strict_downstream_eligible",
            "downstream_eligible",
        ]
        elig_col = find_first_existing_column(summary_work, possible_eligibility_cols)

        if elig_col is not None:
            summary_work["nb10_eligibility_bool"] = to_bool_like(summary_work[elig_col])

            eligible_park_set = set(
                summary_work.loc[
                    summary_work["nb10_eligibility_bool"],
                    PARK_ID_COLUMN
                ].astype(str)
            )

            cohort_contract_summary_rows.append(
                {
                    "check": "eligible_parks_equal_graph_nodes",
                    "status": eligible_park_set == graph_node_set,
                    "details": (
                        "PASS"
                        if eligible_park_set == graph_node_set
                        else f"graph_only={len(graph_node_set - eligible_park_set)}, "
                             f"eligible_only={len(eligible_park_set - graph_node_set)}"
                    ),
                }
            )

            audit_source_notes.append(
                f"Fallback eligibility cross-check used summary artifact "
                f"with park column `{summary_park_col}` and eligibility column `{elig_col}`."
            )
        else:
            cohort_contract_summary_rows.append(
                {
                    "check": "eligible_parks_equal_graph_nodes",
                    "status": None,
                    "details": "summary artifact has park ids, but no eligibility-like column was found",
                }
            )
            audit_source_notes.append(
                "Summary artifact has park ids, αλλά δεν βρέθηκε eligibility-like column."
            )
    else:
        cohort_contract_summary_rows.append(
            {
                "check": "eligible_parks_equal_graph_nodes",
                "status": None,
                "details": "summary artifact exists but is not suitable for park-level set equality",
            }
        )
        audit_source_notes.append(
            "Summary artifact exists, αλλά δεν έχει park-id-like column, άρα έγινε skip."
        )

else:
    cohort_contract_summary_rows.append(
        {
            "check": "eligible_parks_equal_graph_nodes",
            "status": None,
            "details": "no upstream eligibility-compatible artifact found -> check skipped",
        }
    )
    audit_source_notes.append(
        "No upstream eligibility-compatible artifact found."
    )

# ============================================================
# D. Optional heavy final-dataset key cross-check
# ============================================================

if RUN_HEAVY_FINAL_DATASET_KEY_CROSSCHECK and OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"].exists():
    final_keys_df = safe_load_csv(
        OPTIONAL_ARTIFACT_PATHS["final_feature_engineered_dataset"],
        "final_feature_engineered_dataset",
        dtype_map={PARK_ID_COLUMN: "string"},
        usecols=[PARK_ID_COLUMN, TIMESTAMP_COLUMN],
    )

    final_keys_df[PARK_ID_COLUMN] = standardize_park_id(final_keys_df[PARK_ID_COLUMN])
    final_keys_df[TIMESTAMP_COLUMN] = parse_timestamp_strict(final_keys_df[TIMESTAMP_COLUMN], TIMESTAMP_COLUMN)
    assert_no_duplicate_keys(final_keys_df, [PARK_ID_COLUMN, TIMESTAMP_COLUMN], "final_feature_engineered_dataset")

    split_keys_df = pd.concat(
        [
            train_df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN]],
            val_df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN]],
            test_df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN]],
        ],
        axis=0,
        ignore_index=True,
    )

    missing_keys = split_keys_df.merge(
        final_keys_df,
        on=[PARK_ID_COLUMN, TIMESTAMP_COLUMN],
        how="left",
        indicator=True,
    )
    missing_keys = missing_keys.loc[missing_keys["_merge"] == "left_only"]

    cohort_contract_summary_rows.append(
        {
            "check": "all_split_keys_exist_in_final_engineered_dataset",
            "status": len(missing_keys) == 0,
            "details": "PASS" if len(missing_keys) == 0 else f"missing_keys={len(missing_keys)}",
        }
    )
else:
    cohort_contract_summary_rows.append(
        {
            "check": "all_split_keys_exist_in_final_engineered_dataset",
            "status": None,
            "details": "heavy check skipped by policy",
        }
    )

cohort_contract_summary_df = pd.DataFrame(cohort_contract_summary_rows)
upstream_audit_notes_df = pd.DataFrame({"note": audit_source_notes})

print("Optional upstream cross-checks completed.")
display(cohort_contract_summary_df)
display(upstream_audit_notes_df)

Distance-matrix audit completed.


,distance_shape,distance_min,distance_max,diagonal_near_zero,symmetric_ok
0,"(256, 256)",0.000000,878.771463,True,True


Feature-role manifest created.


,column,role,present_in_train,numeric_in_train
0,park_id,backbone_control,True,False
1,timestamp,backbone_control,True,False
2,test_flag,backbone_control,True,True
3,Power_Output_Normalized,target,True,True
4,Baseline_Prediction,control_baseline,True,True
5,turbine,candidate_feature,True,False
6,hub_height_m,candidate_feature,True,True
7,rotor_diameter_m,candidate_feature,True,True
8,nominal_power_kW,candidate_feature,True,True
9,lat,candidate_feature,True,True


Optional upstream cross-checks completed.


,check,status,details
0,split_union_equals_graph_nodes,True,PASS
1,eligible_parks_equal_graph_nodes,True,PASS
2,all_split_keys_exist_in_final_engineered_dataset,None,heavy check skipped by policy


,note
0,Eligibility cross-check used `nb02_meta_covera...


## Compact exports και τελικό verification summary

Το `NB10` πρέπει να εξάγει μόνο compact readiness artifacts και όχι νέα μεγάλα row-level datasets.

Βασικά exports:

- `nb10_node_index_map.csv`
- `nb10_feature_role_manifest.csv`
- `nb10_split_graph_contract_summary.csv`
- `nb10_artifact_status_manifest.csv`
- `nb10_graph_artifact_summary.csv`
- `nb10_cohort_contract_summary.csv`
- `nb10_distance_matrix_summary.csv`
- `nb10_upstream_audit_notes.csv`

Σημείωση export policy:

- το notebook μπορεί να κάνει overwrite μόνο στα compact NB10 exports
- δεν πειράζει τα upstream canonical processed CSVs

Η τελική summary cell πρέπει να απαντά καθαρά:

- ποια checks πέρασαν
- ποια checks έγιναν skipped by design
- αν το observed test-start των local artifacts είναι συμβατό με την `2019-12-01` reference day
- και ότι το notebook **δεν** άλλαξε benchmark reporting, **δεν** έκανε training, **δεν** εισήγαγε νέο model claim

In [9]:
# ============================================================
# Export compact outputs
# ============================================================

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_targets = {
    "nb10_node_index_map.csv": node_index_map_df,
    "nb10_feature_role_manifest.csv": feature_role_manifest_df,
    "nb10_split_graph_contract_summary.csv": split_graph_contract_summary_df,
    "nb10_artifact_status_manifest.csv": artifact_status_manifest_df,
    "nb10_graph_artifact_summary.csv": graph_artifact_summary_df,
    "nb10_cohort_contract_summary.csv": cohort_contract_summary_df,
    "nb10_distance_matrix_summary.csv": distance_summary_df,
    "nb10_upstream_audit_notes.csv": upstream_audit_notes_df,
}

for filename, df in export_targets.items():
    out_path = EXPORT_DIR / filename
    df.to_csv(out_path, index=False)

print("Compact NB10 exports written successfully.")
for filename in export_targets:
    print("-", EXPORT_DIR / filename)

# ============================================================
# Final verification summary
# ============================================================


observed_test_start = pd.Timestamp(
    split_graph_contract_summary_df.loc[
        split_graph_contract_summary_df["split"] == "test", "timestamp_min"
    ].iloc[0]
)
vogt_reference_day = pd.Timestamp("2019-12-01 00:00:00")
observed_test_start_matches_vogt_day = bool(
    observed_test_start.normalize() == vogt_reference_day
)

final_summary_rows = [
    {
        "check": "primary_artifacts_loaded",
        "status": True,
        "details": "train / val / test / graph_node_order / edge_index / distance_matrix loaded",
    },
    {
        "check": "node_order_valid",
        "status": True,
        "details": f"n_nodes={n_nodes}, duplicate_park_ids=0",
    },
    {
        "check": "split_schema_identical",
        "status": True,
        "details": "train / val / test share identical schema",
    },
    {
        "check": "split_overlap_absent",
        "status": True,
        "details": "no (park_id, timestamp) overlap across splits",
    },
    {
        "check": "split_rows_mappable_to_graph_nodes",
        "status": bool(split_graph_contract_summary_df["all_rows_mappable_to_node_index"].all()),
        "details": "all split rows map to valid node_idx",
    },
    {
        "check": "edge_index_in_range",
        "status": (graph_artifact_summary_df["edge_index_min"].iloc[0] >= 0) and (
            graph_artifact_summary_df["edge_index_max"].iloc[0] < n_nodes
        ),
        "details": f"min={graph_artifact_summary_df['edge_index_min'].iloc[0]}, max={graph_artifact_summary_df['edge_index_max'].iloc[0]}",
    },
    {
        "check": "distance_matrix_consistent",
        "status": bool(distance_summary_df["diagonal_near_zero"].iloc[0] and distance_summary_df["symmetric_ok"].iloc[0]),
        "details": "square / symmetric / near-zero diagonal",
    },
    {
        "check": "observed_test_start_matches_vogt_day",
        "status": observed_test_start_matches_vogt_day,
        "details": (
            f"observed_test_start={observed_test_start}, reference_day={vogt_reference_day.date()}"
        ),
    },
    {
        "check": "benchmark_reporting_untouched",
        "status": True,
        "details": "NB10 does not modify canonical benchmark reporting",
    },
    {
        "check": "model_training_performed",
        "status": False,
        "details": "expected False by notebook scope",
    },
]

final_verification_df = pd.DataFrame(final_summary_rows)

display(final_verification_df)

# Hard checks που δεν επιτρέπεται να αποτύχουν.
hard_fail_rows = final_verification_df[
    final_verification_df["check"].isin(
        [
            "primary_artifacts_loaded",
            "node_order_valid",
            "split_schema_identical",
            "split_overlap_absent",
            "split_rows_mappable_to_graph_nodes",
            "edge_index_in_range",
            "distance_matrix_consistent",
        ]
    )
]

all_hard_checks_ok = bool(hard_fail_rows["status"].all())

if all_hard_checks_ok:
    print("\nNB10 CONTRACT CHECKS PASSED")
    print("Το notebook είναι verification-first scaffold για future graph-model work.")
    print(
        "Observed test-start alignment with Vogt reference day:",
        observed_test_start_matches_vogt_day,
        f"(observed={observed_test_start})",
    )
else:
    raise AssertionError(
        "Κάποιο hard contract check απέτυχε. "
        "Δες τα compact manifests και το final_verification_df."
    )

Compact NB10 exports written successfully.
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_node_index_map.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_feature_role_manifest.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_split_graph_contract_summary.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_artifact_status_manifest.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_graph_artifact_summary.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_cohort_contract_summary.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\nb10_graph_contract\nb10_distance_matrix_summary.csv
- C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\diagnostics\

,check,status,details
0,primary_artifacts_loaded,True,train / val / test / graph_node_order / edge_i...
1,node_order_valid,True,"n_nodes=256, duplicate_park_ids=0"
2,split_schema_identical,True,train / val / test share identical schema
3,split_overlap_absent,True,"no (park_id, timestamp) overlap across splits"
4,split_rows_mappable_to_graph_nodes,True,all split rows map to valid node_idx
5,edge_index_in_range,True,"min=0, max=255"
6,distance_matrix_consistent,True,square / symmetric / near-zero diagonal
7,observed_test_start_matches_vogt_day,True,"observed_test_start=2019-12-01 01:00:00, refer..."
8,benchmark_reporting_untouched,True,NB10 does not modify canonical benchmark repor...
9,model_training_performed,False,expected False by notebook scope



NB10 CONTRACT CHECKS PASSED
Το notebook είναι verification-first scaffold για future graph-model work.
Observed test-start alignment with Vogt reference day: True (observed=2019-12-01 01:00:00)


## Συμπέρασμα

Το `NB10` ολοκλήρωσε το πρώτο repository-ready implementation pass ως:

**graph data-interface / split-to-graph contract / artifact-verification notebook**

Στο τρέχον scope:

- δεν έγινε training
- δεν έγινε benchmark update
- δεν δημιουργήθηκε νέο predictive model
- δεν έγινε PHM overclaiming

Το notebook παρέχει τώρα ένα καθαρό handoff layer για μελλοντικό graph-model implementation issue, με explicit:

- `park_id` handling
- node-order verification
- split-to-graph compatibility checks
- graph artifact integrity audit
- compact readiness manifests

Αν στο current local rerun το observed test-start παραμείνει γύρω στην `2019-12-01`,
αυτό πρέπει να διαβαστεί ως **verification finding συμβατό με το dataset paper** και όχι ως κάτι που το notebook διορθώνει ή αναδιατυπώνει τεχνητά.